In [ ]:
import cv2
import numpy as np
import json
import os
import ast
import shutil

# ============================================================
# GLOBAL VARIABLES
# ============================================================

target_colors = []
detected_obstacles_list = []
final_click = None
original_image = None

# ============================================================
# ROBUST NESTED READER (Parser for map_obstacles.txt)
# ============================================================

def read_obstacles_txt_robust(txt_path):
    """
    Reads and parses a text file containing obstacle data.
    Extracts physical dimensions (width, height, aspect ratio) and calculates the center X.
    """
    physical_obstacles = []

    if not os.path.exists(txt_path):
        print(f"[PARSER ERROR] File does not exist: {txt_path}")
        return physical_obstacles

    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        obstacles_dict = {}
        block_counter = 0

        for line in lines:
            # Clean up comments and whitespace
            line = line.split('//')[0].split('#')[0].strip()
            if not line:
                continue
            
            # Increment block counter to avoid merging separate identical objects
            if "/* OBSTACLE */" in line or "/* END_OBSTACLE */" in line:
                block_counter += 1
                continue

            if line.endswith(','):
                line = line[:-1]

            # Replace infinite values and unnecessary labels
            line = line.replace("'inf'", "1e9").replace('"inf"', "1e9").replace("inf", "1e9").replace("x=", "")

            try:
                data = ast.literal_eval(line)
                if isinstance(data, list) and len(data) >= 5:
                    x_range = data[2]
                    y_range = data[3]
                    desc = data[4]

                    # Unique key to avoid merging symmetric objects with the same name
                    unique_key = f"{desc}__block_{block_counter}"

                    if isinstance(x_range, list) and isinstance(y_range, list):
                        if unique_key not in obstacles_dict:
                            obstacles_dict[unique_key] = []
                        obstacles_dict[unique_key].append(data)
            except Exception:
                pass

        # Calculate bounding boxes and dimensions for each grouped obstacle
        for unique_key, segments in obstacles_dict.items():
            min_x_global = float("inf")
            max_x_global = float("-inf")
            min_y_global = float("inf")
            max_y_global = float("-inf")

            valid_segments = 0
            for seg in segments:
                x_range = seg[2]
                y_range = seg[3]
                
                # Ignore infinite bounds (e.g. infinite walls) for physical geometry calculation
                if max(x_range) > 100000 or max(y_range) > 100000:
                    continue
                    
                min_x_global = min(min_x_global, min(x_range))
                max_x_global = max(max_x_global, max(x_range))
                min_y_global = min(min_y_global, min(y_range))
                max_y_global = max(max_y_global, max(y_range))
                valid_segments += 1

            if valid_segments == 0:
                continue # It was an infinite wall, skip it for shape matching

            physical_width = max_x_global - min_x_global
            physical_height = max_y_global - min_y_global
            center_x = (min_x_global + max_x_global) / 2.0
            
            # Clean up the real name
            real_name = unique_key.split("__block_")[0]

            physical_obstacles.append({
                "name": real_name,
                "num_sides": len(segments),
                "width_m": physical_width,
                "height_m": physical_height,
                "aspect_ratio": physical_width / physical_height if physical_height > 0 else 1.0,
                "exact_segments": segments,
                "center_x": center_x
            })
            print(f"[TXT PARSER] Loaded obstacle '{real_name}': {physical_width:.2f}m x {physical_height:.2f}m")

    except Exception as e:
        print(f"[GLOBAL PARSER ERROR] Failed to process file: {e}")

    return physical_obstacles

# ============================================================
# GEOMETRY AND TOPOLOGICAL SIGNATURE FUNCTIONS
# ============================================================

def extract_proportion_signature(vertices):
    """
    Calculates the aspect ratio and relative distance proportions 
    between consecutive vertices of a contour.
    """
    points = [(v[0][0], v[0][1]) for v in vertices]
    n = len(points)
    if n < 3:
        return 1.0, []

    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    bw = max(xs) - min(xs)
    bh = max(ys) - min(ys)
    aspect_ratio = bw / bh if bh > 0 else 1.0

    # Normalized relative distances between consecutive vertices
    distances = []
    perimeter = 0.0
    for i in range(n):
        p1 = points[i]
        p2 = points[(i + 1) % n]
        d = np.hypot(p2[0] - p1[0], p2[1] - p1[1])
        distances.append(d)
        perimeter += d

    proportions = [d / perimeter for d in distances] if perimeter > 0 else []
    return aspect_ratio, proportions

# ============================================================
# MOUSE INTERACTION CALLBACKS
# ============================================================

def select_multiple_colors(event, x, y, flags, param):
    """
    Callback for Phase 1: Records target colors based on user clicks on the image.
    """
    global target_colors, original_image
    original_image = param
    if event != cv2.EVENT_LBUTTONDOWN:
        return

    orig_h, orig_w = original_image.shape[:2]
    orig_x = int(x * (orig_w / 900))
    orig_y = int(y * (orig_h / 450))

    if 0 <= orig_x < orig_w and 0 <= orig_y < orig_h:
        bgr_val = original_image[orig_y, orig_x]
        target_colors.append(bgr_val.astype(np.float32))
        print(f"[COLOR PICKER] Color {bgr_val.tolist()} registered. Total: {len(target_colors)}")

def on_mouse_final_selection(event, x, y, flags, param):
    """
    Callback for Phase 2 & 3: Records the user's click coordinates for object selection.
    """
    global final_click
    if event != cv2.EVENT_LBUTTONDOWN:
        return

    orig_w, orig_h = param
    final_click = (int(x * (orig_w / 900)), int(y * (orig_h / 450)))

# ============================================================
# MAIN PIPELINE (3-PHASE STATE MACHINE)
# ============================================================

def process_navgrid_pipeline(image_path, json_path, txt_path):
    global target_colors, detected_obstacles_list, final_click

    print("\n[PIPELINE] Starting vision engine and topological validation...")
    target_colors.clear()
    detected_obstacles_list.clear()
    final_click = None

    img = cv2.imread(image_path)
    if img is None:
        print(f"[ERROR] Could not load image at: {image_path}")
        return

    # CROP FIELD
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([0, 0, 200], dtype=np.uint8), np.array([180, 30, 255], dtype=np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return
    x, y, w, h = cv2.boundingRect(sorted(contours, key=cv2.contourArea, reverse=True)[0])
    cropped_field = img[y:y + h, x:x + w]
    field_h, field_w = cropped_field.shape[:2]

    # ========================================================
    # PHASE 1: COLOR SELECTION
    # ========================================================
    original_view = cv2.resize(cropped_field, (900, 450))
    phase1_name = "PHASE 1: Click colors -> Press P"
    cv2.imshow(phase1_name, original_view)
    cv2.setMouseCallback(phase1_name, select_multiple_colors, cropped_field)

    while True:
        key = cv2.waitKey(1) & 0xFF
        if key == ord("p") or key == ord("P"):
            if len(target_colors) > 0: break
        elif key == 27: # ESC
            cv2.destroyAllWindows()
            return
    cv2.destroyAllWindows()

    # SEGMENTATION AND MORPHOLOGY
    print("[PIPELINE] Isolating color masses...")
    field_float = cropped_field.astype(np.float32)
    final_mask = np.zeros(cropped_field.shape[:2], dtype=bool)

    for color_ref in target_colors:
        dist = np.sqrt(np.sum((field_float - color_ref) ** 2, axis=2))
        final_mask = np.logical_or(final_mask, dist < 35.0)

    white_mask = (final_mask * 255).astype(np.uint8)
    broken_mask = cv2.morphologyEx(white_mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    generalized_mask = cv2.morphologyEx(broken_mask, cv2.MORPH_CLOSE, np.ones((45, 45), np.uint8))

    contours, _ = cv2.findContours(generalized_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    vector_image = cropped_field.copy()
    magenta_bg = np.full_like(cropped_field, [255, 0, 255])
    magenta_bg[generalized_mask == 255] = [255, 255, 255]

    for contour in contours:
        if cv2.contourArea(contour) < 200: continue
        exact_vertices = cv2.approxPolyDP(contour, 0.015 * cv2.arcLength(contour, True), True)
        x_bb, y_bb, w_bb, h_bb = cv2.boundingRect(exact_vertices)
        ar, props = extract_proportion_signature(exact_vertices)

        detected_obstacles_list.append({
            "vertices": exact_vertices,
            "bbox": (x_bb, y_bb, w_bb, h_bb),
            "aspect_ratio": ar,
            "proportions": props,
            "id": len(detected_obstacles_list),
        })
        cv2.drawContours(vector_image, [exact_vertices], 0, (0, 0, 255), 2)

    # PHYSICAL DATABASE
    print(f"\n[PIPELINE] Analyzing physical DB at: {txt_path}")
    physical_obstacles_txt = read_obstacles_txt_robust(txt_path)

    # ========================================================
    # PHASE 2: CALIBRATION & SHAPE MATCHING (Proportions & Similarity)
    # ========================================================
    viewer_name = "Collision Viewer"
    cv2.imshow("Contrast Map", cv2.resize(magenta_bg, (900, 450)))
    cv2.imshow(viewer_name, cv2.resize(vector_image, (900, 450)))
    cv2.setMouseCallback(viewer_name, on_mouse_final_selection, param=(field_w, field_h))

    print("\n--- PHASE 2 INSTRUCTIONS: CALIBRATION ---")
    print("1. Click on the reference obstacle in the image.")
    print("2. Press 'P' to validate its geometric signature and calibrate uniform scale.\n")

    final_uniform_scale = 0
    mapping_reference_name = "Generic_Obstacle"
    output_route = os.path.join(os.path.dirname(txt_path), "map_obstacles_2.txt")
    selected_obs = -1

    while True:
        key = cv2.waitKey(1) & 0xFF

        if final_click is not None:
            cx, cy = final_click
            selected_obs = -1

            for i, obs in enumerate(detected_obstacles_list):
                bx, by, bw, bh = obs["bbox"]
                if bx <= cx <= bx + bw and by <= cy <= by + bh:
                    selected_obs = i
                    break

            temp_image = vector_image.copy()
            if selected_obs != -1:
                cv2.drawContours(temp_image, [detected_obstacles_list[selected_obs]["vertices"]], 0, (0, 255, 0), 4)
            cv2.imshow(viewer_name, cv2.resize(temp_image, (900, 450)))
            final_click = None

        if key == ord("p") or key == ord("P"):
            if selected_obs != -1:
                obs = detected_obstacles_list[selected_obs]
                bx, by, bw, bh = obs["bbox"]
                det_ar = obs["aspect_ratio"]
                best_match = None
                
                # Calculate costs for all TXT obstacles
                matches_with_costs = []
                for obs_txt in physical_obstacles_txt:
                    txt_ar = obs_txt["aspect_ratio"]
                    ar_cost = abs(det_ar - txt_ar) / max(det_ar, txt_ar)
                    matches_with_costs.append((ar_cost, obs_txt))
                
                if matches_with_costs:
                    min_cost = min(c for c, o in matches_with_costs)
                    # Strict tolerance for considering ties
                    best_matches = [o for c, o in matches_with_costs if abs(c - min_cost) < 0.05]
                    
                    if min_cost < 0.3:
                        print(f"\n[INFO] Found {len(best_matches)} object(s) in the TXT with nearly identical proportions:")
                        for idx, bm in enumerate(best_matches):
                            print(f"   -> Candidate {idx+1}: '{bm['name']}' (Center X: {bm['center_x']:.2f})")

                        # ALLIANCE SELECTION (Tie breaker directly on CV2 Window)
                        if len(best_matches) > 1:
                            print("\n[!] AMBIGUITY DETECTED: Ensure the OpenCV image window is focused.")
                            print("Press 'B' (Blue Alliance) to discard the rightmost obstacle (greater X).")
                            print("Press 'R' (Red Alliance) to discard the leftmost obstacle (smaller X).")
                            
                            while True:
                                alliance_key = cv2.waitKey(0) & 0xFF
                                if alliance_key in [ord('b'), ord('B')]:
                                    best_match = min(best_matches, key=lambda o: o["center_x"])
                                    print(f"[ALLIANCE] Blue selected. Kept obstacle with smaller X (X={best_match['center_x']:.2f}).")
                                    break
                                elif alliance_key in [ord('r'), ord('R')]:
                                    best_match = max(best_matches, key=lambda o: o["center_x"])
                                    print(f"[ALLIANCE] Red selected. Kept obstacle with greater X (X={best_match['center_x']:.2f}).")
                                    break
                        else:
                            best_match = best_matches[0]
                            print("[ALLIANCE] Only one viable candidate found, no ambiguity.")

                        if best_match is not None:
                            mapping_reference_name = best_match["name"]
                            scale_x = best_match["width_m"] / bw if bw > 0 else 0
                            scale_y = best_match["height_m"] / bh if bh > 0 else 0
                            final_uniform_scale = (scale_x + scale_y) / 2.0

                            print("\n" + "=" * 60)
                            print(f"[TOPOLOGICAL MATCH SUCCESS] Object: '{mapping_reference_name}'")
                            print(f"[UNIFORM CALIBRATION] Optimal scale: {final_uniform_scale:.6f} m/px (Structural cost: {min_cost:.3f})")
                            
                            shutil.copy(txt_path, output_route)
                            print(f"[SYSTEM] Base file cloned at: {output_route}")
                            print("=" * 60)
                            break
                else:
                    print("\n[FAIL] Block geometry does not match the database.")
            else:
                print("[!] Select an obstacle first.")
        elif key == 27: # ESC
            cv2.destroyAllWindows()
            return

    # ========================================================
    # PHASE 3: TOPOLOGICAL MAPPING AND CANONICAL ALIGNMENT
    # ========================================================
    print("\n--- PHASE 3 INSTRUCTIONS: INFINITE MAPPING ---")
    print("1. Click on ANY obstacle of the same type -> Press 'P'.")
    print(f"2. It will be mapped with strict geometric correction to {output_route}.")
    print("3. Press ESC when you are finished.\n")

    selected_obs = -1
    final_click = None

    while True:
        key = cv2.waitKey(1) & 0xFF

        if final_click is not None:
            cx, cy = final_click
            selected_obs = -1

            for i, obs in enumerate(detected_obstacles_list):
                bx, by, bw, bh = obs["bbox"]
                if bx <= cx <= bx + bw and by <= cy <= by + bh:
                    selected_obs = i
                    break

            temp_image = vector_image.copy()
            if selected_obs != -1:
                cv2.drawContours(temp_image, [detected_obstacles_list[selected_obs]["vertices"]], 0, (255, 255, 0), 4)
            cv2.imshow(viewer_name, cv2.resize(temp_image, (900, 450)))
            final_click = None

        if key == ord("p") or key == ord("P"):
            if selected_obs != -1:
                obs = detected_obstacles_list[selected_obs]
                vertices = obs["vertices"]
                id_obs = obs["id"]

                points_px = [(v[0][0], v[0][1]) for v in vertices]
                n = len(points_px)

                with open(output_route, "a", encoding="utf-8") as f:
                    f.write("\n/* OBSTACLE */\n")
                    
                    for j in range(n):
                        p1_px = points_px[j]
                        p2_px = points_px[(j + 1) % n]

                        # COORDINATE TRANSFORMATION (AdvantageScope Inverted-Y)
                        x1_m = p1_px[0] * final_uniform_scale
                        y1_m = (field_h - p1_px[1]) * final_uniform_scale
                        
                        x2_m = p2_px[0] * final_uniform_scale
                        y2_m = (field_h - p2_px[1]) * final_uniform_scale

                        min_x, max_x = min(x1_m, x2_m), max(x1_m, x2_m)
                        min_y, max_y = min(y1_m, y2_m), max(y1_m, y2_m)

                        # JAVA REGEX COMPATIBLE WRITING
                        if abs(x2_m - x1_m) < 0.001:
                            # Vertical Line
                            f.write(f"[0.0, {round(x1_m, 4)}, [{round(min_x, 4)}, {round(max_x, 4)}], [{round(min_y, 4)}, {round(max_y, 4)}], \"{mapping_reference_name}\"]\n")
                        elif abs(y2_m - y1_m) < 0.001:
                            # Horizontal Line
                            f.write(f"[0.0, {round(y1_m, 4)}, [{round(min_x, 4)}, {round(max_x, 4)}], [{round(min_y, 4)}, {round(max_y, 4)}], \"{mapping_reference_name}\"]\n")
                        else:
                            # Diagonal Line
                            m = (y2_m - y1_m) / (x2_m - x1_m)
                            b = y1_m - (m * x1_m)
                            f.write(f"[{round(m, 4)}, {round(b, 4)}, [{round(min_x, 4)}, {round(max_x, 4)}], [{round(min_y, 4)}, {round(max_y, 4)}], \"{mapping_reference_name}\"]\n")

                    f.write("/* END_OBSTACLE */\n")

                print(f"[MAPPED] Obstacle ID:{id_obs} registered and validated as '{mapping_reference_name}' ({n} vertices).")

                cv2.drawContours(vector_image, [vertices], 0, (0, 255, 0), 4)
                cv2.imshow(viewer_name, cv2.resize(vector_image, (900, 450)))
                selected_obs = -1
            else:
                print("[!] Select an obstacle first.")

        elif key == 27: # ESC
            break

    cv2.destroyAllWindows()
    print("\n[SYSTEM] Topological mapping and geometric validation successfully completed.")

if __name__ == "__main__":
    img_path = r"C:\Users\HP ENVY\Downloads\field26.png"
    nav_path = r"C:\Users\HP ENVY\Downloads\FRCAI\AlphaFRC\src\main\deploy\pathplanner\navgrid.json"
    txt_path = r"C:\Downloads\map_obstacles.txt"

    process_navgrid_pipeline(img_path, nav_path, txt_path)


[PIPELINE] Starting vision engine and topological validation...
[COLOR PICKER] Color [110, 13, 13] registered. Total: 1
[COLOR PICKER] Color [22, 22, 114] registered. Total: 2
[PIPELINE] Isolating color masses...

[PIPELINE] Analyzing physical DB at: C:\Downloads\map_obstacles.txt
[TXT PARSER] Loaded obstacle '2025 - Reefscape Welded': 1.66m x 1.92m
[TXT PARSER] Loaded obstacle '2026 - Rebuilt Welded': 1.21m x 1.21m
[TXT PARSER] Loaded obstacle '2026 - Rebuilt Welded': 1.21m x 1.21m
[TXT PARSER] Loaded obstacle '2026 - Rebuilt Welded': 1.20m x 0.31m
[TXT PARSER] Loaded obstacle '2026 - Rebuilt Welded': 1.20m x 0.31m
[TXT PARSER] Loaded obstacle '2026 - Rebuilt Welded': 1.20m x 0.31m
[TXT PARSER] Loaded obstacle '2026 - Rebuilt Welded': 1.20m x 0.31m

--- PHASE 2 INSTRUCTIONS: CALIBRATION ---
1. Click on the reference obstacle in the image.
2. Press 'P' to validate its geometric signature and calibrate uniform scale.


[INFO] Found 2 object(s) in the TXT with nearly identical proportio